# Converting an XML Corpus to a Plaintext Corpus on Disk

This template is like [load_xml.ipynb](https://github.com/langeslag/ehtc/blob/main/templates/load_xml.ipynb) except it preserves rubrics, and it stores each document to disk as a plaintext file.

In [1]:
from pathlib import Path
from lxml import etree
from git import Repo

We'll ascertain the ECHOE repository has been cloned so we have XML documents to work with:

In [2]:
# HTTPS clone point:
remote = 'https://github.com/ECHOEProject/echoe.git'
# Desired target folder name:
local = Path.cwd().parent / 'corpora' / 'echoe'
# Only clone if the target folder doesn't already exist:
if not(local.exists()):
    repo = Repo.clone_from(remote, local)
# Else, just update the working copy from remote:
else:
    repo = Repo(local)
    assert isinstance(repo, Repo)
    repo.remotes.origin.pull()
assert not repo.bare

In [3]:
# Normalization matrix (Tironian notes are language-dependent and are thus handled separately below):
substitutions = {
    'ę': 'æ',
    'ƿ': 'w',
    'ẏ': 'y',
    'ſ': 's',
    '': 's', # Using the glyph for descending s, instead of the unicode key point
    'v': 'u',
    'j': 'i',
    'ꝛ': 'r',
    '&': 'et',
    '\uf149': 'þ',
    '\ue337': 'þ',
    '·': '',
    ' ': '',
    '\n': '',
    '\u2028': ''
}

# Token normalization:
def normalize(token):
    # Lowercase:
    token = token.lower()
    for k,v in substitutions.items():
        # Carry out replacements:
        token = token.replace(k, v)
    return token

# Discarding unwanted elements:
def simplify(branch):
    discard = ['abbr', 'am', 'sic', 'del', 'note', 'surplus', 'orig', 'fw']  
    # Now we define their text nodes as empty strings:
    query = ['{http://www.tei-c.org/ns/1.0}' + i for i in discard]
    for hit in branch.iter(query):
        for element in hit.iter():
            element.text = ''
            element.tail = ''
    return branch

In [4]:
parser = etree.XMLParser(remove_blank_text=True,resolve_entities=True)
corpus_folder = local / 'xml'
plaintext_folder = Path.cwd().parent / 'corpora' / 'echoe-plaintext'
Path(plaintext_folder).mkdir(parents=True, exist_ok=True)
corpus = dict()
for file in corpus_folder.glob('*.xml'):
    basename = file.name[:-4]
    tree = etree.parse(file, parser=parser)
    root = simplify(tree.getroot())
    segments = dict()
    rubric_counter = 0
    for segment in root.iter('{http://www.tei-c.org/ns/1.0}head', '{http://www.tei-c.org/ns/1.0}s'):
        if segment.tag == '{http://www.tei-c.org/ns/1.0}head':
            rubric_counter += 1
            identifier = 'rubric' + str(rubric_counter)
        else:
            identifier = segment.get('{http://www.w3.org/XML/1998/namespace}id')
        tokens = []
        for token in segment.iter('{http://www.tei-c.org/ns/1.0}w'):
            if token.get('{http://www.w3.org/XML/1998/namespace}lang') == 'la' or token.xpath('ancestor::*[@xml:lang][1]/@xml:lang')[0] == 'la':
                token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'et').replace('⹒', 'et')
            else:
                token_string = normalize(etree.tostring(token, method='text', encoding='unicode')).replace('⁊', 'and').replace('⹒', 'and')
            # If a word element is marked as the last part of a word, add its text content to the preceding token:
            if token.get('part') == 'F':
                position = len(tokens)-1
                tokens[position] = tokens[position] + token_string
            else:
                tokens.append(token_string)
        segments[identifier] = tokens
    corpus[basename] = segments
        

In [5]:
for ref,doc in corpus.items():
    target_file = Path(plaintext_folder / str(ref + '.txt'))
    previous_identifier = ''
    prior_content = False
    with open(target_file, 'w') as f:
        for identifier,tokens in doc.items():
            if 'rubric' in identifier and (prior_content == False or 'rubric' in previous_identifier):
                line = ' '.join(tokens) + '\n\n'
            elif 'rubric' in identifier:
                line = '\n' + ' '.join(tokens) + '\n\n'
            else:
                line = identifier[1:] + ': ' + ' '.join(tokens) + '\n'
            line = line.replace('  ', ' ')
            f.write(line)
            previous_identifier = identifier
            prior_content = True